In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn import svm
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline

An explanation of each column from the source of each dataset.
~~~
1) S1 "Suit of card #1"
    Ordinal (1-4) representing {Hearts, Spades, Diamonds, Clubs}

2) C1 "Rank of card #1"
    Numerical (1-13) representing (Ace, 2, 3, ... , Queen, King)

3) S2 "Suit of card #2"
    Ordinal (1-4) representing {Hearts, Spades, Diamonds, Clubs}

4) C2 "Rank of card #2"
    Numerical (1-13) representing (Ace, 2, 3, ... , Queen, King)

5) S3 "Suit of card #3"
    Ordinal (1-4) representing {Hearts, Spades, Diamonds, Clubs}

6) C3 "Rank of card #3"
    Numerical (1-13) representing (Ace, 2, 3, ... , Queen, King)

7) S4 "Suit of card #4"
    Ordinal (1-4) representing {Hearts, Spades, Diamonds, Clubs}

8) C4 "Rank of card #4"
    Numerical (1-13) representing (Ace, 2, 3, ... , Queen, King)

9) S5 "Suit of card #5"
    Ordinal (1-4) representing {Hearts, Spades, Diamonds, Clubs}

10) C5 "Rank of card 5"
    Numerical (1-13) representing (Ace, 2, 3, ... , Queen, King)

11) CLASS "Poker Hand"
    Ordinal (0-9)

    0: Nothing in hand; not a recognized poker hand 
    1: One pair; one pair of equal ranks within five cards
    2: Two pairs; two pairs of equal ranks within five cards
    3: Three of a kind; three equal ranks within five cards
    4: Straight; five cards, sequentially ranked with no gaps
    5: Flush; five cards with the same suit
    6: Full house; pair + different rank three of a kind
    7: Four of a kind; four equal ranks within five cards
    8: Straight flush; straight + flush
    9: Royal flush; {Ace, King, Queen, Jack, Ten} + flush
~~~

# Preface

### Does the dataset meet the suitability criteria for logistic regression or SVMs given on the assignment page?
Yes. There is a target variable: the 11th variable noting class/type of poker hand. The other variables are categorical noting the suit and rank of each card of the five cards in the hand. 

### How will the prediction work?
For exploration, let's try predicting on three, four, and five cards from the hand.
One and two cards feel too random given that the majority of the hand is unknown. 

# Predicting Hand Types using logistic regression and support vector machines

### Logistic regression

In [ ]:
# The columns of the dataset have no headers, so we'll just make our own using the column explanation provided with the dataset
df = pd.read_csv('data/poker+hand/poker-hand-training-true.data', header=None, names=["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4", "S5", "C5", "CLASS"])
print(df.head())

Are the classes balanced?

In [ ]:
print(df.CLASS.value_counts())
print((df.CLASS.value_counts()[0] + df.CLASS.value_counts()[1])/df.CLASS.value_counts().sum())

Absolutely not. We need to account for this with our class weights when initializing the model.

In [123]:
# Reduce each hand to the first three, four, and five cards
three_card_df = df[["S1", "C1", "S2", "C2", "S3", "C3"]]
four_card_df = df[["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4"]]
five_card_df = df[["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4", "S5", "C5"]]
# Get the classification of each hand
type_df = df.CLASS
print(len(type_df))

weights = pd.Series()
# The class weight of class i is (apparently) equal to (total sample count / (number of classes * number of sample in class i)) 
for x in type_df.value_counts().index: 
    weights[x] = type_df.count() / (type_df.nunique() * type_df.value_counts()[x]) 
print(weights)

# Make tables for results (each has a column for each hand type)
three_results = pd.DataFrame(columns = (x for x in range(10)))
four_results = pd.DataFrame(columns = (x for x in range(10)))
five_results = pd.DataFrame(columns = (x for x in range(10)))
 

25010
0      0.200192
1      0.235966
2      2.073798
3      4.875244
4     26.892473
5     46.314815
6     69.472222
7    416.833333
9    500.200000
8    500.200000
dtype: float64


In [124]:
# A test dataset is provided, so it needs to be prepared.

test_df = pd.read_csv('data/poker+hand/poker-hand-testing.data', header=None, names=["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4", "S5", "C5", "CLASS"])
print(test_df.head())



three_card_test_df = test_df[["S1", "C1", "S2", "C2", "S3", "C3"]]
four_card_test_df = test_df[["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4"]]
five_card_test_df = test_df[["S1", "C1", "S2", "C2", "S3", "C3", "S4", "C4", "S5", "C5"]]

test_type_df = test_df.CLASS
print(len(test_type_df))

   S1  C1  S2  C2  S3  C3  S4  C4  S5  C5  CLASS
0   1   1   1  13   2   4   2   3   1  12      0
1   3  12   3   2   3  11   4   5   2   5      1
2   1   9   4   6   1   4   3   2   3   9      1
3   1   4   3  13   2  13   2   1   3   6      1
4   3  10   2   7   1   2   2  11   4   9      0
1000000


In [ ]:
'''
I need to use one-hot encoding because all my features are categorical and need to be respected as such.
For the sake of simplicity, I'm going to use a pipeline.
We never talked about pipelines in-class, but I learned about them from CS4320.
They'll make one-hot encoding (and passing the encoded features to the model) much easier. 
'''
def make_model(model):
    return make_pipeline(OneHotEncoder(handle_unknown='ignore'), model)

In [ ]:
# An increased number of iterations were given because a warning was given due to the default max iterations (100) not being enough given the size of the dataset.
three_lm = make_model(LogisticRegression(class_weight=weights.to_dict(), max_iter=300))

three_lm.fit(three_card_df, type_df)

# y_pred = three_lm.predict(three_card_test_df)
y_pred = three_lm.predict(three_card_test_df)
p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

three_results.loc[len(three_results)] = f

Logistic regression is very poor on three cards. The precision for invalid poker hands and one-pair hands are good, but the same can't be said about recall (which is horrible for all except for Royal-flush).

What about four cards?

In [ ]:
# The number of iterations needed to be increased further.
four_lm = make_model(LogisticRegression(class_weight=weights.to_dict(), max_iter=500))

four_lm.fit(four_card_df, type_df)

y_pred = four_lm.predict(four_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

four_results.loc[len(four_results)] = f

Better in some spots. Interestingly, the f-score on two-pairs nearly doubled when compared to the regression using three cards. Maybe because two pairs requires four cards? So far, I think logistic regression (at least the way I'm doing it) is not suited for guessing hand type based on partial hands.

Let's try five cards / a full hand:

In [ ]:
# The number of iterations needed to be increased even further.
five_lm = make_model(LogisticRegression(class_weight=weights.to_dict(), max_iter=1500))

five_lm.fit(five_card_df, type_df)

y_pred = five_lm.predict(five_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

five_results.loc[len(five_results)] = f

Still not great. The f-score for Straight-flush is now zero. Either the regression was unable to predict straight flushes, or the precision and recall values got so small they are reported as zero. On the other hand, the f-score for flush and full-house improved very slightly, but the improvements are practically meanless given each f-score's power of ten.  

### Support vector machines

Let's try this with an SVM starting with three cards again.

In [ ]:
three_clf_linear = make_model(svm.SVC(kernel='linear', class_weight=weights.to_dict()))
three_clf_linear.fit(three_card_df, type_df)

y_pred = three_clf_linear.predict(three_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Three Card - SVM (linear kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

three_results.loc[len(three_results)] = f

In [ ]:
three_clf_poly = make_model(svm.SVC(kernel='poly', degree=2, class_weight=weights.to_dict()))
three_clf_poly.fit(three_card_df, type_df)

y_pred = three_clf_poly.predict(three_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Three Card - SVM (polynomial kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

three_results.loc[len(three_results)] = f

\***It's worth noting at this point that both the linear and polynomial models took roughly 6 minutes to train (on my setup).**
There's a scikit-learn extension that uses hardware acceleration (on Intel x86 CPUs) for various tasks, notably ML-related tasks. While testing, that 6 minutes dropped to roughly 30 seconds with no difference in results. I'm not going to have it included and enabled in the final submission, but I will be using it while I work on the project.

In [ ]:
three_clf_rbf = make_model(svm.SVC(kernel='rbf', class_weight=weights.to_dict()))
three_clf_rbf.fit(three_card_df, type_df)

y_pred = three_clf_rbf.predict(three_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Three Card - SVM (radial basis function kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

three_results.loc[len(three_results)] = f

In [ ]:
four_clf_linear = make_model(svm.SVC(kernel='linear', class_weight=weights.to_dict()))
four_clf_linear.fit(four_card_df.values, type_df)

y_pred = four_clf_linear.predict(four_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Four Card - SVM (linear kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

four_results.loc[len(four_results)] = f

In [ ]:
four_clf_poly = make_model(svm.SVC(kernel='poly', degree=2, class_weight=weights.to_dict()))
four_clf_poly.fit(four_card_df, type_df)

y_pred = four_clf_poly.predict(four_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Four Card - SVM (2-degree polynomial kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

four_results.loc[len(four_results)] = f

In [ ]:
four_clf_rbf = make_model(svm.SVC(kernel='rbf', class_weight=weights.to_dict()))
four_clf_rbf.fit(four_card_df, type_df)

y_pred = four_clf_rbf.predict(four_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Four Card - SVM (radial basis function kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

four_results.loc[len(four_results)] = f

In [ ]:
five_clf_linear = make_model(svm.SVC(kernel='linear', class_weight=weights.to_dict()))
five_clf_linear.fit(five_card_df, type_df)

y_pred = five_clf_linear.predict(five_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Five Card - SVM (linear kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

five_results.loc[len(five_results)] = f

In [ ]:
five_clf_poly = make_model(svm.SVC(kernel='poly', degree=2, class_weight=weights.to_dict()))
five_clf_poly.fit(five_card_df, type_df)

y_pred = five_clf_poly.predict(five_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Five Card - SVM (2-degree polynomial kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

five_results.loc[len(five_results)] = f

In [ ]:
five_clf_rbf = make_model(svm.SVC(kernel='rbf', class_weight=weights.to_dict()))
five_clf_rbf.fit(five_card_df.values, type_df)

y_pred = five_clf_rbf.predict(five_card_test_df)

p,r,f,s = precision_recall_fscore_support(test_type_df, y_pred, labels=type_df.value_counts().index.sort_values())
print("Five Card - SVM (radial basis kernel)")
print(f"precision:\n\tInvalid poker hand: {p[0]}\n\tOne-pair: {p[1]}\n\tTwo-pairs: {p[2]}\n\tThree-of-a-kind: {p[3]}\n\tStraight: {p[4]}\n\tFlush: {p[5]}\n\tFull-house: {p[6]}\n\tFour-of-a-kind: {p[7]}\n\tStraight-flush: {p[8]}\n\tRoyal-flush: {p[9]}")
print(f"recall:\n\tInvalid poker hand: {r[0]}\n\tOne-pair: {r[1]}\n\tTwo-pairs: {r[2]}\n\tThree-of-a-kind: {r[3]}\n\tStraight: {r[4]}\n\tFlush: {r[5]}\n\tFull-house: {r[6]}\n\tFour-of-a-kind: {r[7]}\n\tStraight-flush: {r[8]}\n\tRoyal-flush: {r[9]}")
print(f"f-score:\n\tInvalid poker hand: {f[0]}\n\tOne-pair: {f[1]}\n\tTwo-pairs: {f[2]}\n\tThree-of-a-kind: {f[3]}\n\tStraight: {f[4]}\n\tFlush: {f[5]}\n\tFull-house: {f[6]}\n\tFour-of-a-kind: {f[7]}\n\tStraight-flush: {f[8]}\n\tRoyal-flush: {f[9]}")

five_results.loc[len(five_results)] = f

In [121]:
final_threes = three_results.rename(
    index={0: 'LR', 1: 'SVM (linear kernel)', 2: 'SVM (polynomial kernel)', 3: 'SVM (radial basis kernel)'},
    columns={0: 'Invalid Hand', 1: 'One-pair', 2: 'Two-pairs', 3:'Three-of-a-kind', 4: 'Straight', 5:'Flush', 6:'Full-house', 7:'Four-of-a-kind', 8: 'Straight-flush', 9: 'Royal-flush'}
    ).rename_axis('model').copy() 
final_fours = four_results.rename_axis('model').rename(
    index={0: 'LR', 1: 'SVM (linear kernel)', 2: 'SVM (polynomial kernel)', 3: 'SVM (radial basis kernel)'},
    columns={0: 'Invalid Hand', 1: 'One-pair', 2: 'Two-pairs', 3:'Three-of-a-kind', 4: 'Straight', 5:'Flush', 6:'Full-house', 7:'Four-of-a-kind', 8: 'Straight-flush', 9: 'Royal-flush'},
    ).copy()
final_fives = five_results.rename(
    index={0: 'LR', 1: 'SVM (linear kernel)', 2: 'SVM (polynomial kernel)', 3: 'SVM (radial basis kernel)'},
    columns={0: 'Invalid Hand', 1: 'One-pair', 2: 'Two-pairs', 3:'Three-of-a-kind', 4: 'Straight', 5:'Flush', 6:'Full-house', 7:'Four-of-a-kind', 8: 'Straight-flush', 9: 'Royal-flush'}
    ).rename_axis('model').copy() 
print('f-scores for model by hand type (three cards of five from hand used for prediction)')
display(final_threes)
print('f-scores for model by hand type (four cards of five from hand used for prediction)')
display(final_fours)
print('f-scores for model by hand type (five cards/full hand used for prediction)')
display(final_fives)

f-scores for model by hand type (three cards of five from hand used for prediction)


,Invalid Hand,One-pair,Two-pairs,Three-of-a-kind,Straight,Flush,Full-house,Four-of-a-kind,Straight-flush,Royal-flush
model,,,,,,,,,,
LR,0.113627,0.076116,0.061886,0.037494,0.010745,0.004529,0.002940,0.001846,0.0,0.000244
SVM (linear kernel),0.188576,0.132296,0.063649,0.043780,0.010377,0.004821,0.003231,0.002483,0.0,0.000788
SVM (polynomial kernel),0.669855,0.202343,0.167160,0.092910,0.029193,0.062830,0.014973,0.016043,0.0,0.000000
SVM (radial basis kernel),0.596475,0.325001,0.140705,0.096147,0.027501,0.060412,0.013418,0.013514,0.0,0.000000


f-scores for model by hand type (four cards of five from hand used for prediction)


,Invalid Hand,One-pair,Two-pairs,Three-of-a-kind,Straight,Flush,Full-house,Four-of-a-kind,Straight-flush,Royal-flush
model,,,,,,,,,,
LR,0.131811,0.094522,0.069258,0.037893,0.011151,0.004881,0.003197,0.002286,0.0,0.000191
SVM (linear kernel),0.222855,0.127906,0.075571,0.042902,0.011927,0.005310,0.003455,0.004175,0.0,0.001791
SVM (polynomial kernel),0.723432,0.398417,0.185500,0.134765,0.049206,0.273082,0.023105,0.007143,0.0,0.000000
SVM (radial basis kernel),0.620230,0.424614,0.160523,0.161591,0.036657,0.239284,0.014815,0.000000,0.0,0.000000


f-scores for model by hand type (five cards/full hand used for prediction)


,Invalid Hand,One-pair,Two-pairs,Three-of-a-kind,Straight,Flush,Full-house,Four-of-a-kind,Straight-flush,Royal-flush
model,,,,,,,,,,
LR,0.127520,0.122000,0.070563,0.037195,0.012259,0.004657,0.003334,0.001386,0.0,0.000696
SVM (linear kernel),0.236909,0.178672,0.076202,0.043334,0.015014,0.005238,0.003578,0.001826,0.0,0.003413
SVM (polynomial kernel),0.753277,0.581469,0.239959,0.371999,0.077953,0.705731,0.016699,0.000000,0.0,0.000000
SVM (radial basis kernel),0.627349,0.489908,0.192334,0.246454,0.033633,0.531181,0.004155,0.000000,0.0,0.000000


# Answering a select number of exploration questions on Canvas

### Which variables are predictive of the target variable?

Every variable (aside from the target variable). Because the target variable is directly dependent on the suit and rank of each card in hand and the variables describe said suit and rank for each card, every variable is predictive of the target variable.

### Can logistic regression or a linear SVM predict well?

No. Let's use the f-scores for each hand as an indicator of prediction performance: no hand type has achieved an f-score above 0.24 for three or more cards using logistic regression or a linear support vector machine.  

### How generalizable are the different models on your data? How does bias-variance tradeoff affect which model you might choose?

Fairly generalizable. Because the test dataset and the training dataset have distinct hands (though class imbalance is kept the same) and are both so large, the models should perform similarly regardless of testing data.

As for bias-variance tradeoff, testing indicates that linear methods (logistic regression and linear SVMs) aren't suitable, polynomial SVMs perform best (among the tested models), while SVMs using radial basis kernels perform better than linear models, but experience a regression in performance compared to polynomial models despite the increased complexity. 

### Is there a difference between the polynomial and RBF SVMs?

According to performance, yes. For this specific analysis, polynomial SVMs have better accuracy on average than RBF SVMs. However, more analysis is needed to determine why this is.

### What affect does changing the class_weight in an SVM have on your data? How might this be important for this data?

For this dataset specifically, I cannot express how much changing the class weights affects how the models value the data. The distribution in the amount of different hand types in this dataset roughly align with how common the hands are generally when you draw five cards / 'in actual play'. As a result, 92% of the dataset comprises of invalid hands and one-pairs despite having 10 different hand types. Consequently, the models end up just predicting 'invalid hand' or 'one-pair' for any given hand.

### Is there a difference in runtime performance?

Yes. All logistic regression models perform near identically in terms of runtime (with very small time increases as the number of features increases from 3 to 5), while the SVMs vary widely depending on kernel and feature amount. The SVMs used took anywhere from 10 to 20 times (15-30 seconds) as long as logistic regression models (~1.5 seconds) to train. Polynomial kernel SVMs perform faster than linear kernel SVMs when using the same kernel and feature amount (the relative difference grows as feature cont increases), while radial basis function kernel SVMs run 10-13 second slower than polynomial kernels rather than by a difference of percentage.   

### Logistic regression and LinearSVC use one-vs-rest (OVR) for multi-class classification. SVC uses one-vs-one (OVO). Where n is the number of classes, OVR learns n models, whereas OVO learns n(n-1)/2 (n choose 2) models. What effect does this have on performance? 

At 5 cards, the number of classes is 10. For OVR, 10 models are learned. For OVO, 10*(10-1)/2 = 45 models are used. Assuming the question is referring to accuracy/effectiveness-relaed performance rather than runtime performance, the increased number of models generally improves performance given the difference in f-scores for each hand type in the logistic regression models vs the SVM models.